## Example 1: Basic File Download with Caching

Two cells: the first runs against the real endpoint (the test runner skips it, and records the skip, where the network is unreachable); the second is its offline twin against a loopback server, which always runs.

In [ ]:
import tempfile
from pathlib import Path
from fastdownload import FastDownload

print("Example 1: Basic File Download with Caching")

with tempfile.TemporaryDirectory() as temp_dir:
    d = FastDownload(base=Path(temp_dir))
    url = "https://github.com/robots.txt"
    first = d.download(url)  # fetched into the archive directory
    fetched_at = first.stat().st_mtime_ns
    print(f"Downloaded to: {first} ({first.stat().st_size} bytes)")
    second = d.download(url)  # already in the archive, so it is returned without a new request
    print(f"Same path as first download: {first == second}")
    content = first.read_text()
    print(f"First 100 characters: {content[:100]!r}")
    assert first == second == d.arch_path() / "robots.txt", (first, second)
    assert second.stat().st_mtime_ns == fetched_at, "the cached copy was downloaded again"
    assert "User-agent" in content, content[:200]

print("Example 1 completed")

In [ ]:
import functools, http.server, tempfile, threading
from pathlib import Path
from fastdownload import FastDownload

print("Example 1 (offline twin): Basic File Download with Caching")

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    hits = []
    def log_message(self, *args): pass
    def do_GET(self):
        QuietHandler.hits.append(self.path)
        super().do_GET()

with tempfile.TemporaryDirectory() as temp_dir:
    site = Path(temp_dir, "site"); site.mkdir()
    (site / "robots.txt").write_text("User-agent: *\nDisallow: /private/\n")
    server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), functools.partial(QuietHandler, directory=str(site)))
    threading.Thread(target=server.serve_forever, daemon=True).start()
    try:
        url = f"http://127.0.0.1:{server.server_port}/robots.txt"
        d = FastDownload(base=Path(temp_dir, "fd"))
        path1 = d.download(url)
        print(f"Downloaded to: {path1} ({path1.stat().st_size} bytes)")
        path2 = d.download(url)  # already in the archive: served from the cache, no second request
        print(f"Same path as first download: {path1 == path2}; requests served: {QuietHandler.hits}")
        assert path1 == path2 and path1.read_text() == (site / "robots.txt").read_text()
        assert QuietHandler.hits == ["/robots.txt"], QuietHandler.hits
    finally:
        server.shutdown()
        server.server_close()

print("Example 1 offline twin completed")

## Example 2: Cached Copies and Force Refresh

fastdownload 0.0.8 has no custom-file-name parameter: a download is named after the URL's last path component. Two cells: the first runs against the real endpoint (the test runner skips it, and records the skip, where the network is unreachable); the second is its offline twin against a loopback server, which always runs.

In [ ]:
# Example 2: the cached copy wins until `force=True` re-fetches it -- real endpoint
import json
import tempfile
from pathlib import Path
from fastdownload import FastDownload

print("Example 2: Cached Copies and Force Refresh")

with tempfile.TemporaryDirectory() as temp_dir:
    d = FastDownload(base=Path(temp_dir))
    json_url = "https://httpbingo.org/json"
    path = d.download(json_url)
    print(f"Downloaded to: {path} (fastdownload names the file after the URL's last component)")
    original = path.read_text()
    path.write_text('{"modified": "locally"}')
    kept = d.download(json_url).read_text()  # no force: the local copy is returned as it is
    refreshed = d.download(json_url, force=True).read_text()  # force: fetched again
    print(f"Without force: {kept}\nWith force=True: {refreshed[:60]}...")
    assert path.name == "json", path.name
    assert kept == '{"modified": "locally"}' and refreshed == original
    assert "slideshow" in json.loads(refreshed), refreshed[:200]

print("Example 2 completed")

In [ ]:
import functools, http.server, json, tempfile, threading
from pathlib import Path
from fastdownload import FastDownload

print("Example 2 (offline twin): Cached Copies and Force Refresh")

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *args): pass

with tempfile.TemporaryDirectory() as temp_dir:
    site = Path(temp_dir, "site"); site.mkdir()
    (site / "sample_data.json").write_text(json.dumps({"slideshow": {"title": "Sample Slide Show"}}))
    server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), functools.partial(QuietHandler, directory=str(site)))
    threading.Thread(target=server.serve_forever, daemon=True).start()
    try:
        url = f"http://127.0.0.1:{server.server_port}/sample_data.json"
        d = FastDownload(base=Path(temp_dir, "fd"))
        path = d.download(url)
        print(f"Downloaded to: {path} (the file name comes from the URL)")
        original = path.read_text()
        path.write_text('{"modified": "locally"}')
        kept = d.download(url).read_text()
        refreshed = d.download(url, force=True).read_text()
        print(f"Without force: {kept}\nWith force=True: {refreshed}")
        assert path.name == "sample_data.json", path.name
        assert kept == '{"modified": "locally"}' and refreshed == original
        assert json.loads(refreshed)["slideshow"]["title"] == "Sample Slide Show"
    finally:
        server.shutdown()
        server.server_close()

print("Example 2 offline twin completed")

## Example 3: Archive Download and Automatic Extraction

Two cells: the first runs against the real endpoint (the test runner skips it, and records the skip, where the network is unreachable); the second is its offline twin against a loopback server, which always runs.

In [ ]:
# Example 3: download an archive and extract it with `get` -- fastdownload's own README example, real endpoint
import tempfile
from pathlib import Path
from fastdownload import FastDownload

print("Example 3: Archive Download and Automatic Extraction")

with tempfile.TemporaryDirectory() as temp_dir:
    d = FastDownload(base=Path(temp_dir))
    archive_url = "https://s3.amazonaws.com/fast-ai-sample/mnist_tiny.tgz"
    data_dir = d.get(archive_url)  # download into the archive directory, then extract into the data directory
    archive = d.arch_path() / "mnist_tiny.tgz"
    extracted = [p for p in data_dir.rglob("*") if p.is_file()]
    print(f"Archive: {archive} ({archive.stat().st_size} bytes)")
    print(f"Extracted to {data_dir}: {len(extracted)} files, top level {sorted(p.name for p in data_dir.iterdir())}")
    assert data_dir == d.data_path(arch=archive) and data_dir.is_dir(), data_dir
    assert archive.is_file() and extracted, (archive, data_dir)
    assert d.get(archive_url) == data_dir  # already extracted: reused, not downloaded again

print("Example 3 completed")

In [ ]:
import functools, http.server, tarfile, tempfile, threading
from pathlib import Path
from fastdownload import FastDownload

print("Example 3 (offline twin): Archive Download and Automatic Extraction")

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *args): pass

with tempfile.TemporaryDirectory() as temp_dir:
    site = Path(temp_dir, "site"); site.mkdir()
    staging = Path(temp_dir, "staging", "sample"); (staging / "data").mkdir(parents=True)
    (staging / "readme.txt").write_text("This is a sample readme file.\n")
    (staging / "data" / "sample.csv").write_text("name,age,city\nAlice,30,New York\nBob,25,London\n")
    with tarfile.open(site / "sample.tgz", "w:gz") as tar:
        tar.add(staging, arcname="sample")
    server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), functools.partial(QuietHandler, directory=str(site)))
    threading.Thread(target=server.serve_forever, daemon=True).start()
    try:
        d = FastDownload(base=Path(temp_dir, "fd"))
        data_dir = d.get(f"http://127.0.0.1:{server.server_port}/sample.tgz")  # download + extract
        files = sorted(str(p.relative_to(data_dir)) for p in data_dir.rglob("*") if p.is_file())
        print(f"Extracted to {data_dir}: {files}")
        assert files == ["data/sample.csv", "readme.txt"], files
        assert (data_dir / "data" / "sample.csv").read_text().splitlines()[1] == "Alice,30,New York"
        assert d.get(f"http://127.0.0.1:{server.server_port}/sample.tgz") == data_dir  # already extracted: reused
    finally:
        server.shutdown()
        server.server_close()

print("Example 3 offline twin completed")

## Example 4: Progress Tracking and Download Status

Two cells: the first runs against the real endpoint (the test runner skips it, and records the skip, where the network is unreachable); the second is its offline twin against a loopback server, which always runs.

In [ ]:
# Example 4: several downloads with per-file status and timing -- real endpoints
import json
import tempfile
import time
from pathlib import Path
from fastdownload import FastDownload

print("Example 4: Progress Tracking and Download Status")

status_urls = [
    "https://httpbingo.org/json",
    "https://httpbingo.org/base64/SGVsbG8gV29ybGQ%3D",
    "https://github.com/robots.txt",
]
with tempfile.TemporaryDirectory() as temp_dir:
    d = FastDownload(base=Path(temp_dir))
    fetched = {}
    for url in status_urls:
        start = time.perf_counter()
        path = d.download(url)  # a failed download fails this example: nothing is swallowed
        elapsed = time.perf_counter() - start
        fetched[path.name] = path.read_bytes()
        print(f"  ✓ {path.name}: {path.stat().st_size} bytes in {elapsed:.3f}s from {url}")
    print(f"Total: {sum(map(len, fetched.values()))} bytes across {len(fetched)} files")
    assert fetched["SGVsbG8gV29ybGQ%3D"] == b"Hello World", fetched["SGVsbG8gV29ybGQ%3D"]
    assert "slideshow" in json.loads(fetched["json"]) and b"User-agent" in fetched["robots.txt"]

print("Example 4 completed")

In [ ]:
import functools, http.server, os, tempfile, threading, time
from pathlib import Path
from fastdownload import FastDownload

print("Example 4 (offline twin): Progress Tracking and Download Status")

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *args): pass

with tempfile.TemporaryDirectory() as temp_dir:
    site = Path(temp_dir, "site"); site.mkdir()
    sizes = {"small.json": 64, "medium.txt": 64 * 1024, "large.bin": 4 * 1024 * 1024}
    for name, size in sizes.items():
        (site / name).write_bytes(os.urandom(size))
    server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), functools.partial(QuietHandler, directory=str(site)))
    threading.Thread(target=server.serve_forever, daemon=True).start()
    try:
        d = FastDownload(base=Path(temp_dir, "fd"))
        results = []
        for name in sizes:
            start = time.perf_counter()
            path = d.download(f"http://127.0.0.1:{server.server_port}/{name}")
            elapsed = time.perf_counter() - start
            results.append((name, path.stat().st_size, elapsed))
            print(f"  ✓ {name}: {path.stat().st_size} bytes in {elapsed:.3f}s")
            assert path.read_bytes() == (site / name).read_bytes(), name
        print(f"Total: {sum(size for _, size, _ in results)} bytes across {len(results)} files")
        assert [(name, size) for name, size, _ in results] == list(sizes.items())
    finally:
        server.shutdown()
        server.server_close()

print("Example 4 offline twin completed")

## Example 5: Advanced Configuration and Download Management

Two cells: the first runs against the real endpoint (the test runner skips it, and records the skip, where the network is unreachable); the second is its offline twin against a loopback server, which always runs.

In [ ]:
# Example 5: per-purpose download roots, recorded checks, and removal -- real endpoints
import ast
import tempfile
import types
from pathlib import Path
from fastdownload import FastDownload

print("Example 5: Advanced Configuration and Download Management")

category_urls = {
    "data": "https://httpbingo.org/json",
    "cache": "https://github.com/robots.txt",
    "models": "https://httpbingo.org/uuid",  # a fresh UUID on every request
}
with tempfile.TemporaryDirectory() as temp_dir:
    package = Path(temp_dir, "mypkg"); package.mkdir()
    module = types.SimpleNamespace(__file__=str(package / "__init__.py"))  # download_checks.py lives beside it
    configs = {name: FastDownload(base=Path(temp_dir, name), module=module) for name in category_urls}
    for name, url in category_urls.items():
        path = configs[name].download(url)
        configs[name].update(url)  # record the size + md5 of what was fetched
        assert path.parent == configs[name].arch_path() and configs[name].check(url, path), path
        print(f"  ✓ {name}: {path.name} ({path.stat().st_size} bytes) in {configs[name].arch_path()}")
    recorded = ast.literal_eval((package / "download_checks.py").read_text())
    print(f"Recorded checks for {len(recorded)} URLs")
    assert set(recorded) == set(category_urls.values()), recorded
    try:
        configs["models"].download(category_urls["models"], force=True)  # the server now sends different bytes
    except Exception as err:  # fastdownload raises a bare Exception when a recorded check fails
        assert "corrupt or not latest version" in str(err), err
        print(f"Changed remote file rejected: {err}")
    else:
        raise AssertionError("a download that no longer matches its recorded checks was accepted")
    configs["cache"].rm(category_urls["cache"])
    assert not (configs["cache"].arch_path() / "robots.txt").exists()
    print("rm() removed the cached robots.txt")

print("Example 5 completed")

In [ ]:
import functools, http.server, tempfile, threading, types
from pathlib import Path
from fastdownload import FastDownload

print("Example 5 (offline twin): Advanced Configuration and Download Management")

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *args): pass

with tempfile.TemporaryDirectory() as temp_dir:
    site = Path(temp_dir, "site"); site.mkdir()
    (site / "model.bin").write_bytes(b"weights-v1" * 100)
    server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), functools.partial(QuietHandler, directory=str(site)))
    threading.Thread(target=server.serve_forever, daemon=True).start()
    try:
        url = f"http://127.0.0.1:{server.server_port}/model.bin"
        configs = {name: FastDownload(base=Path(temp_dir, name)) for name in ("data", "models", "cache")}
        for name, downloader in configs.items():
            print(f"  - {name}: archives in {downloader.arch_path()}, data in {downloader.data_path()}")
        # A package's `download_checks.py` records size + md5 per URL; `module` says where it lives.
        package = Path(temp_dir, "mypkg"); package.mkdir()
        checked = FastDownload(base=Path(temp_dir, "models"), module=types.SimpleNamespace(__file__=str(package / "__init__.py")))
        path = checked.download(url)
        checked.update(url)
        print(f"Recorded checks: {(package / 'download_checks.py').read_text()}")
        (site / "model.bin").write_bytes(b"tampered")  # the server now serves different bytes
        try:
            checked.download(url, force=True)
        except Exception as err:  # fastdownload raises a bare Exception when a recorded check fails
            assert "corrupt or not latest version" in str(err), err
            print(f"Tampered download rejected: {err}")
        else:
            raise AssertionError("a download that no longer matches its recorded checks was accepted")
        checked.rm(url)
        assert not path.exists(), path
        print("rm() removed the archive")
    finally:
        server.shutdown()
        server.server_close()

print("Example 5 offline twin completed")